<a href="https://colab.research.google.com/github/Didarulisalmdidar/QLSTM_Multilayer_Network/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Cell 1: Mount Google Drive ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── Cell 2: Imports & Config ───────────────────────────────
import os
import numpy as np
import pandas as pd

In [ ]:
# ── Configuration ──
BASE_DIR    = "/content/drive/MyDrive/MSC Thesis"
DATA_DIR    = os.path.join(BASE_DIR, "Data")
OUTPUT_DIR  = os.path.join(BASE_DIR, "Processed")

FILES = {
    "Energy"    : os.path.join(DATA_DIR, "Energy_close_price.csv"),
    "IT"        : os.path.join(DATA_DIR, "IT_close_price.csv"),
    "Financial" : os.path.join(DATA_DIR, "Financial_close_price.csv"),
    "Healthcare": os.path.join(DATA_DIR, "Healthcare_close_price.csv"),
}

WINDOW_SIZE = 5
STRIDE      = 1
OUT_SAMPLE  = 80
N_YEARS     = 20

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Drive mounted. Base dir: {BASE_DIR}")

✅ Drive mounted. Base dir: /content/drive/MyDrive/MSC Thesis


In [ ]:
# ── Helper Functions ───────────────────────────────────────

def compute_log_returns(prices):
    return np.log(prices).diff().dropna()

def normalize(log_ret):
    mean = log_ret.mean(axis=0)
    std  = log_ret.std(axis=0).replace(0, 1)
    return (log_ret - mean) / std

def segment_and_window(norm_ret):
    """
    Returns dict: {year_idx: ndarray of shape (n_windows, WINDOW_SIZE, n_stocks)}
    Auto-calculates in_sample to always produce exactly 20 years.
    """
    values      = norm_ret.values
    total       = len(values)
    in_sample   = (total - OUT_SAMPLE) // N_YEARS   # 247 for 5031 rows
    segment_len = in_sample + OUT_SAMPLE             # 327
    n_windows   = (segment_len - WINDOW_SIZE) // STRIDE + 1  # 323

    segments = {}
    for j in range(N_YEARS):
        start_idx = j * in_sample
        end_idx   = start_idx + segment_len
        if end_idx > total:
            print(f"  [WARN] Year {j}: skipping.")
            break
        seg = values[start_idx:end_idx]
        segments[j] = np.stack(
            [seg[i : i + WINDOW_SIZE] for i in range(0, n_windows, STRIDE)],
            axis=0,
        ).astype(np.float32)   # (323, 5, 15)

    return segments, in_sample, n_windows

In [ ]:
def save_year_as_csv(arr, tickers, sector_dir, year_idx):
    """
    Saves one year's windowed data as CSV.
    arr shape: (n_windows, WINDOW_SIZE, n_stocks)

    CSV structure:
        window_id | day | AAPL | MSFT | ...
        0         | 0   | ...
        0         | 1   | ...
        ...
        0         | 4   | ...
        1         | 0   | ...
        ...
    """
    n_windows, window_size, n_stocks = arr.shape
    rows = []

    for w in range(n_windows):
        for d in range(window_size):
            row = {"window_id": w, "day": d}
            for s, ticker in enumerate(tickers):
                row[ticker] = arr[w, d, s]
            rows.append(row)

    df = pd.DataFrame(rows)
    out_path = os.path.join(sector_dir, f"year_{year_idx:02d}.csv")
    df.to_csv(out_path, index=False)
    return out_path


In [ ]:
# ── Main Pipeline ──────────────────────────────────────────

print("=" * 60)
print(" Preprocessing Pipeline (Full CSV with Segmentation)")
print("=" * 60)

summary = {}

for sector, filepath in FILES.items():
    print(f"\n── {sector} ──────────────────────────")

    # Load
    prices  = pd.read_csv(filepath, index_col=0, parse_dates=True)
    tickers = list(prices.columns)
    print(f"  Loaded   : {prices.shape[0]} days × {prices.shape[1]} stocks")

    # Log returns + normalize
    log_ret  = compute_log_returns(prices)
    norm_ret = normalize(log_ret)
    print(f"  Log ret  : {log_ret.shape}")
    print(f"  Norm     : mean={norm_ret.mean().mean():.4f}  std={norm_ret.std().mean():.4f}")

    # Segment + window
    segments, in_sample, n_windows = segment_and_window(norm_ret)
    print(f"  Segments : {len(segments)} years  |  shape: {segments[0].shape}")
    print(f"  in_sample={in_sample}  segment_len={in_sample+OUT_SAMPLE}  windows/year={n_windows}")

    # Save each year as CSV
    sector_dir = os.path.join(OUTPUT_DIR, sector)
    os.makedirs(sector_dir, exist_ok=True)

    for j, arr in segments.items():
        save_year_as_csv(arr, tickers, sector_dir, j)

    print(f"  Saved    : {len(segments)} CSV files → {sector_dir}/")
    print(f"             year_00.csv (2005) ... year_{len(segments)-1:02d}.csv (2024)")

    summary[sector] = {
        "years"     : len(segments),
        "shape"     : segments[0].shape,
        "in_sample" : in_sample,
        "n_windows" : n_windows,
    }


# ── Summary ────────────────────────────────────────────────

print(f"\n{'='*60}")
print(" Summary")
print(f"{'='*60}")
for sector, info in summary.items():
    print(f"  {sector:15s}: {info['years']} years | shape: {info['shape']} | windows/year: {info['n_windows']}")

# Quick verification — load one file back and check
print(f"\n── Quick Verification ──────────────────────────────")
test_path = os.path.join(OUTPUT_DIR, "Energy", "year_00.csv")
test_df   = pd.read_csv(test_path)
n_windows_check = test_df["window_id"].nunique()
print(f"  Energy/year_00.csv")
print(f"  Shape          : {test_df.shape}  (rows={test_df.shape[0]}, cols={test_df.shape[1]})")
print(f"  Unique windows : {n_windows_check}  ← should be {summary['Energy']['n_windows']}")
print(f"  Columns        : {list(test_df.columns)}")
print(f"  NaN count      : {test_df.isna().sum().sum()}")

print(f"\n✅ Done! All files saved to: {OUTPUT_DIR}")
print(f"\n── How to load in training ──────────────────────────")
print(f"  import pandas as pd")
print(f"  import numpy as np")
print(f"  df  = pd.read_csv('Energy/year_00.csv')")
print(f"  arr = df.drop(columns=['window_id','day']).values.reshape(-1, 5, 15)")
print(f"  print(arr.shape)  # (323, 5, 15)")

 Preprocessing Pipeline (Full CSV with Segmentation)

── Energy ──────────────────────────
  Loaded   : 5032 days × 15 stocks
  Log ret  : (5031, 15)
  Norm     : mean=-0.0000  std=1.0000
  Segments : 20 years  |  shape: (323, 5, 15)
  in_sample=247  segment_len=327  windows/year=323
  Saved    : 20 CSV files → /content/drive/MyDrive/MSC Thesis/Processed/Energy/
             year_00.csv (2005) ... year_19.csv (2024)

── IT ──────────────────────────
  Loaded   : 5032 days × 15 stocks
  Log ret  : (5031, 15)
  Norm     : mean=0.0000  std=1.0000
  Segments : 20 years  |  shape: (323, 5, 15)
  in_sample=247  segment_len=327  windows/year=323
  Saved    : 20 CSV files → /content/drive/MyDrive/MSC Thesis/Processed/IT/
             year_00.csv (2005) ... year_19.csv (2024)

── Financial ──────────────────────────
  Loaded   : 5032 days × 15 stocks
  Log ret  : (5031, 15)
  Norm     : mean=-0.0000  std=1.0000
  Segments : 20 years  |  shape: (323, 5, 15)
  in_sample=247  segment_len=327  wind